[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/EN/Seminar_Notebooks/chapter12_seminar_notebook.ipynb)

---

# Chapter 12 Seminar: Spectral Analysis — Practice

**Course:** Time Series Analysis and Forecasting  
**Program:** Bachelor program, Faculty of Cybernetics, Statistics and Economic Informatics, Bucharest University of Economic Studies, Romania  
**Academic Year:** 2025-2026

---

## Seminar Objectives

In this practical seminar, you will:
1. Compute and interpret periodograms and spectral density estimates
2. Extract business cycles from GDP data using HP, BK, and CF filters
3. Measure frequency-specific co-movement using squared coherence
4. Apply wavelet analysis to capture time-varying spectral structure in financial data

**Instructions:**
- Fill in every `# YOUR CODE HERE` section
- Run all cells in order before comparing with the solutions
- Reference solutions are provided at the end of each exercise

## Setup

In [ ]:
# Install required packages (for Colab)
import sys
if 'google.colab' in sys.modules:
    !pip install pywavelets statsmodels scipy yfinance -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy import signal
from scipy.fft import fft, fftfreq
import statsmodels.api as sm
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.filters.bk_filter import bkfilter
from statsmodels.tsa.filters.cf_filter import cffilter

try:
    import pywt
    HAS_PYWT = True
except ImportError:
    HAS_PYWT = False
    print('PyWavelets not installed. Install with: pip install PyWavelets')

try:
    import yfinance as yf
    HAS_YF = True
except ImportError:
    HAS_YF = False
    print('yfinance not installed. Install with: pip install yfinance')

# Course color scheme
COLORS = {
    'blue':       '#1A3A6E',
    'red':        '#DC3545',
    'green':      '#2E7D32',
    'amber':      '#B5853F',
    'orange':     '#E6802E',
    'purple':     '#8E44AD',
    'gray':       '#666666',
    'light_blue': '#5B8BD4',
}

plt.rcParams.update({
    'figure.figsize':    (12, 5),
    'font.size':         11,
    'axes.facecolor':    'none',
    'figure.facecolor':  'none',
    'axes.grid':         False,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    False,
})

print('Setup complete!')
print(f'PyWavelets available: {HAS_PYWT}')
print(f'yfinance available:   {HAS_YF}')

---
## Exercise 1: Periodicities in Sunspot Data

The Wolf sunspot series records annual sunspot counts since 1700 and is one of the longest  
instrumental records in science. The solar magnetic cycle produces an approximately **11-year**  
oscillation — one of the most precisely documented cycles in nature.

**Tasks:**
1. Load the sunspot data and plot the time series
2. Compute the raw periodogram and identify the dominant cycle period (in years)
3. Apply Welch's method and compare with the raw periodogram
4. Overlay a vertical line at the dominant frequency and report the cycle period

### 1.1 Load Data

In [ ]:
# Load sunspot data from statsmodels
sunspots_data = sm.datasets.sunspots.load_pandas()
sunspots      = sunspots_data.data
sunspots.set_index('YEAR', inplace=True)
sunspots.index = sunspots.index.astype(int)

y_sun = sunspots['SUNACTIVITY'].values
T_sun = len(y_sun)
print(f'Observations: {T_sun}')
print(f'Period: {sunspots.index[0]}-{sunspots.index[-1]}')
print(sunspots.head())

### Task 1a: Plot the Sunspot Time Series

Plot `SUNACTIVITY` over time. Use `fill_between` to shade the area under the curve. Label your axes.

In [ ]:
# Task 1a: Plot the sunspot time series
# YOUR CODE HERE

fig, ax = plt.subplots(figsize=(13, 4))
# ax.fill_between(sunspots.index, y_sun, ...)  <-- fill this in
# ax.plot(sunspots.index, y_sun, ...)          <-- fill this in
ax.set_xlabel('Year')
ax.set_ylabel('Sunspot Number')
ax.set_title('Annual Sunspot Activity')
plt.tight_layout()
plt.show()

### Task 1b: Compute the Raw Periodogram

Use `scipy.signal.periodogram` with `fs=1.0` (annual sampling, so frequency is in cycles/year).  
- Skip the DC component (index 0, frequency = 0)
- Find the index of the maximum power
- Report the dominant frequency and the corresponding period in years

In [ ]:
# Task 1b: Compute and plot the raw periodogram
# YOUR CODE HERE

# freqs_p, power_p = signal.periodogram(y_sun, fs=1.0)

# Find the dominant peak (skip DC at freq index 0)
# peak_idx    = np.argmax(power_p[1:]) + 1
# peak_freq   = freqs_p[peak_idx]
# peak_period = 1.0 / peak_freq
# print(f'Dominant period: {peak_period:.1f} years')

fig, ax = plt.subplots(figsize=(12, 5))
# ax.semilogy(freqs_p[1:], power_p[1:], ...)  <-- fill this in
# ax.axvline(peak_freq, ..., label=f'Peak: {peak_period:.1f} years')
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Spectral Density (log scale)')
ax.set_title('Raw Periodogram — Sunspot Data')
plt.tight_layout()
plt.show()

### Task 1c: Apply Welch's Method

Use `scipy.signal.welch` with:
- `fs=1.0`
- `nperseg=64`
- `noverlap=32`
- `window='hann'`

Plot the Welch PSD alongside the raw periodogram. Which one is smoother and why?

In [ ]:
# Task 1c: Welch's method and comparison
# YOUR CODE HERE

# freqs_w, psd_w = signal.welch(y_sun, fs=1.0, nperseg=64, noverlap=32, window='hann')

fig, ax = plt.subplots(figsize=(12, 5))
# ax.semilogy(freqs_p[1:], power_p[1:], ..., label='Raw periodogram')
# ax.semilogy(freqs_w[1:], psd_w[1:],   ..., label="Welch's method")
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Spectral Density (log scale)')
ax.set_title("Periodogram vs Welch's Method — Sunspot Data")
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()

### Solution 1 (reference)

> Attempt all tasks before reading the solution below.

In [ ]:
# ── SOLUTION 1 ────────────────────────────────────────────────

# 1a: Time series
fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(sunspots.index, y_sun, color=COLORS['amber'], alpha=0.4)
ax.plot(sunspots.index, y_sun, color=COLORS['amber'], lw=1.0)
ax.set_xlabel('Year')
ax.set_ylabel('Sunspot Number')
ax.set_title('Annual Sunspot Activity')
plt.tight_layout()
plt.show()

# 1b: Raw periodogram
freqs_p, power_p = signal.periodogram(y_sun, fs=1.0)
peak_idx    = np.argmax(power_p[1:]) + 1
peak_freq   = freqs_p[peak_idx]
peak_period = 1.0 / peak_freq
print(f'Dominant period: {peak_period:.1f} years  (f = {peak_freq:.4f} cycles/year)')

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freqs_p[1:], power_p[1:], color=COLORS['gray'],
            lw=0.8, alpha=0.7, label='Raw periodogram')
ax.axvline(peak_freq, color=COLORS['red'], lw=1.5, ls='--',
           label=f'Peak: {peak_period:.1f} years')
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Spectral Density (log scale)')
ax.set_title('Raw Periodogram — Sunspot Data')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()

# 1c: Welch's method
freqs_w, psd_w = signal.welch(y_sun, fs=1.0, nperseg=64,
                               noverlap=32, window='hann')

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freqs_p[1:], power_p[1:], color=COLORS['gray'],
            lw=0.8, alpha=0.5, label='Raw periodogram')
ax.semilogy(freqs_w[1:], psd_w[1:],   color=COLORS['blue'],
            lw=2.0, label="Welch's method (nperseg=64)")
ax.axvline(peak_freq, color=COLORS['red'], lw=1.5, ls='--',
           label=f'~{peak_period:.0f}-year cycle')
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Spectral Density (log scale)')
ax.set_title("Periodogram vs Welch's Method — Sunspot Data")
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.tight_layout()
plt.show()

print()
print('Key findings:')
print(f'  Dominant solar cycle: ~{peak_period:.0f} years')
print(f'  Welch smooths the noisy raw periodogram by averaging overlapping segments')
print(f'  Both clearly identify the ~11-year peak at f = {peak_freq:.3f} cycles/year')

---
## Exercise 2: GDP Business Cycle Extraction

Business cycles are the recurrent expansions and contractions in economic activity around trend.  
The NBER convention defines business cycle frequencies as **6 to 32 quarters** (1.5-8 years).

**Tasks:**
1. Simulate quarterly log-GDP data with a trend and business cycle component
2. Apply the **HP filter** with $\lambda = 1600$ and plot trend and cycle
3. Apply the **Baxter-King (BK) filter** (6-32 quarters, K=12)
4. Apply the **Christiano-Fitzgerald (CF) filter** (6-32 quarters)
5. Overlay all three cycle estimates and compute the spectral density of the HP cycle

### 2.1 Simulate Quarterly GDP

In [ ]:
# Simulate quarterly log-GDP: trend + business cycle + irregular
np.random.seed(42)
Q        = 200
quarters = pd.date_range('1975Q1', periods=Q, freq='QS')

trend   = 8.0 + 0.007 * np.arange(Q)                         # linear trend
cycle   = 0.02 * np.sin(2 * np.pi * np.arange(Q) / 20)       # 20-quarter cycle
irr     = np.cumsum(np.random.normal(0, 0.004, Q))            # random walk noise
log_gdp = pd.Series(trend + cycle + irr, index=quarters, name='log_GDP')

print(f'Simulated log-GDP: {Q} quarterly observations')
print(log_gdp.describe().round(4))

### Task 2a: HP Filter

Apply `hpfilter(log_gdp, lamb=1600)`. It returns `(cycle, trend)`.  
Plot the original series and HP trend in the upper panel, and the HP cycle in the lower panel.

In [ ]:
# Task 2a: HP filter
# YOUR CODE HERE

# hp_cycle, hp_trend = hpfilter(log_gdp, lamb=1600)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# axes[0]: Plot log_gdp and hp_trend
# axes[1]: Plot hp_cycle

for ax in axes:
    ax.axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
plt.tight_layout()
plt.show()

### Task 2b: Baxter-King Filter

Apply `bkfilter(log_gdp, low=6, high=32, K=12)`.  
Note: BK produces `NaN` at the first and last `K=12` observations — plot only the valid range.

In [ ]:
# Task 2b: Baxter-King filter
# YOUR CODE HERE

# bk_cycle = bkfilter(log_gdp, low=6, high=32, K=12)

fig, ax = plt.subplots(figsize=(13, 4))
# ax.plot(bk_cycle.index, bk_cycle.values, ...)  <-- fill this in
ax.axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
ax.set_xlabel('Date')
ax.set_ylabel('Cycle')
ax.set_title('Baxter-King Band-Pass Cycle (6-32 quarters, K=12)')
plt.tight_layout()
plt.show()

### Task 2c: Christiano-Fitzgerald Filter

Apply `cffilter(log_gdp, low=6, high=32, drift=True)`.  
It returns `(cycle, trend)`. Plot only the cycle component.

In [ ]:
# Task 2c: Christiano-Fitzgerald filter
# YOUR CODE HERE

# cf_cycle, cf_trend = cffilter(log_gdp, low=6, high=32, drift=True)

fig, ax = plt.subplots(figsize=(13, 4))
# ax.plot(cf_cycle.index, cf_cycle.values, ...)  <-- fill this in
ax.axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
ax.set_xlabel('Date')
ax.set_ylabel('Cycle')
ax.set_title('Christiano-Fitzgerald Band-Pass Cycle (6-32 quarters)')
plt.tight_layout()
plt.show()

### Task 2d: Compare All Three Cycles

Plot the HP cycle, BK cycle, and CF cycle on the same axes with different colors.

In [ ]:
# Task 2d: Compare all three cycles on one plot
# YOUR CODE HERE

fig, ax = plt.subplots(figsize=(13, 5))
# ax.plot(hp_cycle.index, hp_cycle.values, ..., label='HP cycle')
# ax.plot(bk_cycle.index, bk_cycle.values, ..., label='BK cycle (6-32q)')
# ax.plot(cf_cycle.index, cf_cycle.values, ..., label='CF cycle (6-32q)')
ax.axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
ax.set_xlabel('Date')
ax.set_ylabel('Cycle component')
ax.set_title('Business Cycle Comparison: HP vs BK vs CF')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.tight_layout()
plt.show()

### Task 2e: Spectral Density of the HP Cycle

Compute and plot the periodogram of the HP cycle. What is the dominant period?  
Add vertical lines at `f = 1/32` and `f = 1/6` to mark the NBER business cycle band.

In [ ]:
# Task 2e: Spectral density of the HP cycle
# YOUR CODE HERE

# hp_cycle_vals = hp_cycle.dropna().values
# freqs_c, psd_c = signal.periodogram(hp_cycle_vals, fs=1.0)

fig, ax = plt.subplots(figsize=(12, 5))
# ax.semilogy(freqs_c[1:], psd_c[1:], ...)  <-- fill this in
# ax.axvline(1/32, ..., label='f=1/32 (32 quarters)')
# ax.axvline(1/6,  ..., label='f=1/6  (6 quarters)')
ax.set_xlabel('Frequency (cycles/quarter)')
ax.set_ylabel('Spectral Density (log scale)')
ax.set_title('Periodogram of HP Cycle')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.tight_layout()
plt.show()

### Solution 2 (reference)

> Attempt all tasks before reading the solution below.

In [ ]:
# ── SOLUTION 2 ────────────────────────────────────────────────

# 2a: HP filter
hp_cycle, hp_trend = hpfilter(log_gdp, lamb=1600)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(log_gdp.index, log_gdp.values, color=COLORS['blue'],
             lw=1.5, label='Log GDP')
axes[0].plot(hp_trend.index, hp_trend.values, color=COLORS['red'],
             lw=2.0, ls='--', label='HP trend (lambda=1600)')
axes[0].set_ylabel('Log GDP')
axes[0].set_title('HP Filter: Trend Extraction')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)

axes[1].plot(hp_cycle.index, hp_cycle.values, color=COLORS['blue'], lw=1.5)
axes[1].axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Cycle')
axes[1].set_title('HP Cycle Component')
plt.tight_layout()
plt.show()

# 2b: Baxter-King
bk_cycle = bkfilter(log_gdp, low=6, high=32, K=12)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(bk_cycle.index, bk_cycle.values, color=COLORS['green'], lw=1.5,
        label='BK cycle')
ax.axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
ax.set_xlabel('Date')
ax.set_ylabel('Cycle')
ax.set_title('Baxter-King Band-Pass Cycle (6-32 quarters, K=12)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=1)
plt.tight_layout()
plt.show()

# 2c: Christiano-Fitzgerald
cf_cycle, cf_trend = cffilter(log_gdp, low=6, high=32, drift=True)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(cf_cycle.index, cf_cycle.values, color=COLORS['orange'], lw=1.5,
        label='CF cycle')
ax.axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
ax.set_xlabel('Date')
ax.set_ylabel('Cycle')
ax.set_title('Christiano-Fitzgerald Cycle (6-32 quarters)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=1)
plt.tight_layout()
plt.show()

# 2d: Comparison
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(hp_cycle.index, hp_cycle.values, color=COLORS['blue'],
        lw=1.5, label='HP cycle')
ax.plot(bk_cycle.index, bk_cycle.values, color=COLORS['green'],
        lw=1.5, ls='--', label='BK cycle (6-32q)')
ax.plot(cf_cycle.index, cf_cycle.values, color=COLORS['orange'],
        lw=1.5, ls=':', label='CF cycle (6-32q)')
ax.axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
ax.set_xlabel('Date')
ax.set_ylabel('Cycle component')
ax.set_title('Business Cycle Comparison: HP vs BK vs CF')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.tight_layout()
plt.show()

# 2e: Spectral density of HP cycle
hp_cycle_vals = hp_cycle.dropna().values
freqs_c, psd_c = signal.periodogram(hp_cycle_vals, fs=1.0)
peak_idx_c  = np.argmax(psd_c[1:]) + 1
peak_freq_c = freqs_c[peak_idx_c]
peak_per_c  = 1.0 / peak_freq_c

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freqs_c[1:], psd_c[1:], color=COLORS['blue'], lw=1.5)
ax.axvline(peak_freq_c, color=COLORS['red'], lw=1.5, ls='--',
           label=f'Peak: {peak_per_c:.1f} quarters')
ax.axvline(1/32, color=COLORS['gray'],  lw=1.0, ls=':',  label='f=1/32 (32q)')
ax.axvline(1/6,  color=COLORS['amber'], lw=1.0, ls='--', label='f=1/6  (6q)')
ax.set_xlabel('Frequency (cycles/quarter)')
ax.set_ylabel('Spectral Density (log scale)')
ax.set_title('Periodogram of HP Cycle')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.tight_layout()
plt.show()

print(f'Dominant cycle in HP-filtered GDP: {peak_per_c:.1f} quarters = {peak_per_c/4:.1f} years')
print()
print('Notes:')
print('  - BK and CF filters are more precise at isolating the 6-32 quarter band')
print('  - HP filter has a gradual gain that passes some very-low-frequency noise')
print('  - All three filters recover the simulated 20-quarter (5-year) business cycle')

---
## Exercise 3: Coherence Between Two Economic Series

**Squared coherence** measures the degree of linear association between two time series  
at each frequency — analogous to $R^2$ in regression:

$$C^2_{xy}(\omega) = \frac{|f_{xy}(\omega)|^2}{f_x(\omega)\, f_y(\omega)} \in [0, 1]$$

**Tasks:**
1. Plot the two simulated series (GDP cycle and investment cycle)
2. Compute and plot the squared coherence; mark the known cycle frequencies
3. Extract the phase spectrum and interpret the lead-lag relationship

### 3.1 Generate Two Related Series

In [ ]:
# Simulate GDP and investment cycles with a known lead-lag structure
np.random.seed(77)
N = 400   # 100 years of quarterly data

# GDP cycle: dominant 20-quarter period, secondary 8-quarter period
gdp_cycle_sim = (2.0 * np.sin(2 * np.pi * np.arange(N) / 20) +
                  0.8 * np.sin(2 * np.pi * np.arange(N) / 8) +
                  np.random.normal(0, 0.4, N))

# Investment cycle: lags GDP by 3 quarters at the 20-quarter frequency
lead_inv = 3
inv_cycle_sim = (1.5 * np.sin(2 * np.pi * (np.arange(N) - lead_inv) / 20) +
                  0.6 * np.sin(2 * np.pi * np.arange(N) / 8) +
                  np.random.normal(0, 0.5, N))

print(f'Generated {N} quarterly observations')
print(f'GDP cycle: 20-quarter + 8-quarter + noise')
print(f'Investment: lags GDP by {lead_inv} quarters at the 20-quarter frequency')

### Task 3a: Plot Both Series

In [ ]:
# Task 3a: Plot both series on the same axes
# YOUR CODE HERE

fig, ax = plt.subplots(figsize=(13, 4))
# ax.plot(gdp_cycle_sim, ..., label='GDP cycle')
# ax.plot(inv_cycle_sim, ..., label='Investment cycle')
ax.set_xlabel('Quarter')
ax.set_ylabel('Amplitude')
ax.set_title('Simulated GDP and Investment Cycles')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()

### Task 3b: Compute and Plot Squared Coherence

Use `scipy.signal.coherence` with:
- `fs=4.0` (quarterly series: 4 quarters per year)
- `nperseg=128`, `noverlap=64`

Add vertical dashed lines at `f = 4/20` (20-quarter cycle) and `f = 4/8` (8-quarter cycle).

In [ ]:
# Task 3b: Squared coherence
# YOUR CODE HERE

# freqs_coh, Cxy = signal.coherence(gdp_cycle_sim, inv_cycle_sim,
#                                    fs=4.0, nperseg=128, noverlap=64)

fig, ax = plt.subplots(figsize=(12, 5))
# ax.plot(freqs_coh, Cxy, ...)  <-- fill this in
# ax.axvline(4/20, ..., label='f=4/20 (20-quarter cycle)')
# ax.axvline(4/8,  ..., label='f=4/8  (8-quarter cycle)')
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Squared Coherence')
ax.set_title('Squared Coherence: GDP vs Investment Cycle')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()

### Task 3c: Phase Spectrum

Use `scipy.signal.csd` with the same parameters, then `np.angle(Pxy, deg=True)`.  
At the dominant 20-quarter frequency, what is the phase lead/lag in degrees?  
Convert to quarters: lag = phase / 360 * period.

In [ ]:
# Task 3c: Phase spectrum
# YOUR CODE HERE

# freqs_csd, Pxy = signal.csd(gdp_cycle_sim, inv_cycle_sim,
#                              fs=4.0, nperseg=128, noverlap=64)
# phase = np.angle(Pxy, deg=True)

fig, ax = plt.subplots(figsize=(12, 5))
# ax.plot(freqs_csd, phase, ...)  <-- fill this in
ax.axhline(0, color=COLORS['gray'], ls=':', lw=1.0)
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Phase (degrees)')
ax.set_title('Phase Spectrum: GDP leads Investment')
plt.tight_layout()
plt.show()

# Implied lag at 20-quarter cycle
# expected_phase = 360 * lead_inv / 20
# print(f'Expected phase: {expected_phase} degrees = {lead_inv} quarters lag')

### Solution 3 (reference)

> Attempt all tasks before reading the solution below.

In [ ]:
# ── SOLUTION 3 ────────────────────────────────────────────────

# 3a: Plot both series
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(gdp_cycle_sim, color=COLORS['blue'],   lw=1.5, label='GDP cycle')
ax.plot(inv_cycle_sim, color=COLORS['orange'], lw=1.5, alpha=0.8,
        label='Investment cycle')
ax.set_xlabel('Quarter')
ax.set_ylabel('Amplitude')
ax.set_title('Simulated GDP and Investment Cycles')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()

# 3b: Squared coherence
freqs_coh, Cxy = signal.coherence(gdp_cycle_sim, inv_cycle_sim,
                                   fs=4.0, nperseg=128, noverlap=64)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freqs_coh, Cxy, color=COLORS['blue'], lw=2.0)
ax.axvline(4/20, color=COLORS['red'],    ls='--', lw=1.5,
           label='f=4/20 (20-quarter cycle)')
ax.axvline(4/8,  color=COLORS['orange'], ls='--', lw=1.5,
           label='f=4/8  (8-quarter cycle)')
ax.axhline(0.5,  color=COLORS['gray'],   ls=':',  lw=1.0)
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Squared Coherence')
ax.set_title('Squared Coherence: GDP vs Investment Cycle')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()

# 3c: Phase spectrum
freqs_csd, Pxy = signal.csd(gdp_cycle_sim, inv_cycle_sim,
                             fs=4.0, nperseg=128, noverlap=64)
phase = np.angle(Pxy, deg=True)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freqs_csd, phase, color=COLORS['purple'], lw=1.5)
ax.axvline(4/20, color=COLORS['red'], ls='--', lw=1.5,
           label='f=4/20 (20-quarter cycle)')
ax.axhline(0, color=COLORS['gray'], ls=':', lw=1.0)
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Phase (degrees)')
ax.set_title('Phase Spectrum: GDP leads Investment')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=1)
plt.tight_layout()
plt.show()

# Interpret
expected_phase  = 360.0 * lead_inv / 20
measured_phase  = float(np.interp(4.0/20, freqs_csd, phase))
implied_lag     = abs(measured_phase) / 360 * 20
print(f'Expected phase at 20-quarter cycle: {expected_phase:.1f} degrees')
print(f'Measured phase:                      {measured_phase:.1f} degrees')
print(f'Implied lag:                         {implied_lag:.1f} quarters')
print()
print('Interpretation:')
print('  - High coherence (>0.8) at both the 20-quarter and 8-quarter cycles')
print('  - Phase ~54 degrees at 20-quarter cycle confirms ~3-quarter lag')
print('  - Investment is a lagging indicator of the GDP cycle in this simulation')

---
## Exercise 4: Wavelet Analysis of a Financial Series

Unlike Fourier analysis, **wavelets reveal how frequency content changes over time** —  
essential for financial series where volatility is not constant (volatility clustering).

**Tasks:**
1. Download SPY weekly prices using `yfinance` (or use the simulated series)
2. Compute log-returns and plot the series
3. Apply the Continuous Wavelet Transform with the complex Morlet wavelet
4. Plot the scalogram (wavelet power vs time and period)
5. Identify periods of high volatility in specific frequency bands

### 4.1 Load Financial Data

In [ ]:
# Download SPY data or use a GARCH-simulated substitute
if HAS_YF:
    try:
        ticker     = yf.Ticker('SPY')  # S&P 500 ETF
        price_data = ticker.history(start='2000-01-01', end='2024-12-31')['Close']
        price_wk   = price_data.resample('W').last().dropna()
        log_ret    = np.log(price_wk).diff().dropna()
        dates      = log_ret.index
        returns    = log_ret.values
        print(f'Downloaded SPY: {len(returns)} weekly log-returns')
        print(f'Period: {dates[0].date()} to {dates[-1].date()}')
    except Exception as e:
        print(f'yfinance failed ({e}). Using simulated data.')
        HAS_YF = False

if not HAS_YF:
    # GARCH(1,1)-simulated weekly returns with two crisis episodes
    np.random.seed(99)
    N_fin   = 1200   # ~23 years of weekly data
    sigma   = np.ones(N_fin) * 0.01
    returns = np.zeros(N_fin)
    for t in range(1, N_fin):
        sigma[t]   = np.sqrt(0.00005 + 0.08 * returns[t-1]**2 + 0.91 * sigma[t-1]**2)
        returns[t] = sigma[t] * np.random.standard_normal()
    # Inject two crisis episodes
    returns[400:450] *= 4.0
    returns[900:930] *= 5.0
    dates = pd.date_range('2000-01-07', periods=N_fin, freq='W')
    print(f'Simulated {N_fin} weekly log-returns (2000-2023)')

print(f'Return stats: mean={returns.mean():.5f}, std={returns.std():.5f}')

### Task 4a: Plot the Return Series

Plot the weekly log-returns. Note any periods of unusually high volatility.

In [ ]:
# Task 4a: Plot the return series
# YOUR CODE HERE

fig, ax = plt.subplots(figsize=(13, 4))
# ax.plot(dates, returns, ...)  <-- fill this in
ax.axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
ax.set_xlabel('Date')
ax.set_ylabel('Log Return')
ax.set_title('Weekly Log-Returns')
plt.tight_layout()
plt.show()

### Task 4b: Continuous Wavelet Transform

Use `pywt.cwt` with:
- `wavelet = 'cmor1.5-1.0'` (complex Morlet: bandwidth=1.5, center frequency=1.0)
- `sampling_period = 1/52` (weeks expressed in years)
- Frequencies from 52/104 to 52/4 cycles/year (periods of 4 to 104 weeks)

Convert frequencies to scales using `pywt.frequency2scale(wavelet, freqs_fin)`.  
Then plot the scalogram as `contourf(dates, log2(periods_in_weeks), log2(power))`.

In [ ]:
# Task 4b: CWT and scalogram
# YOUR CODE HERE

if HAS_PYWT:
    wavelet = 'cmor1.5-1.0'

    # Define frequency range (cycles per year), then convert to scales
    # freqs_fin  = np.geomspace(52/104, 52/4, 60)   # 4 to 104 weeks
    # scales_fin = pywt.frequency2scale(wavelet, freqs_fin)

    # Apply CWT
    # coeffs_fin, freqs_cwt_fin = pywt.cwt(returns, scales_fin, wavelet,
    #                                        sampling_period=1/52)
    # power_fin   = np.abs(coeffs_fin) ** 2
    # periods_fin = 1.0 / freqs_cwt_fin   # in years

    fig, axes = plt.subplots(2, 1, figsize=(14, 9),
                              gridspec_kw={'height_ratios': [1, 2.5]})
    # Top: return series
    # axes[0].plot(dates, returns, ...)
    axes[0].set_title('Weekly Returns and Wavelet Power Spectrum')

    # Bottom: scalogram
    # im = axes[1].contourf(dates,
    #                        np.log2(periods_fin * 52),   # convert years to weeks
    #                        np.log2(power_fin + 1e-10),
    #                        levels=20, cmap='YlOrRd')
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Period (weeks, log2 scale)')

    # Y-axis ticks
    yticks_w = [4, 8, 13, 26, 52, 104]
    # axes[1].set_yticks(np.log2(yticks_w))
    # axes[1].set_yticklabels([str(w) + 'w' for w in yticks_w])

    # plt.colorbar(im, ax=axes[1], label='log2(Power)')
    plt.tight_layout()
    plt.show()
else:
    print('PyWavelets not installed. Install with: pip install PyWavelets')

### Task 4c: Interpretation

Edit this cell to answer the following questions based on your scalogram:

1. **Which frequency bands (period ranges) show the highest wavelet power?**  
   *Your answer here.*

2. **Can you identify specific dates when power is particularly high at short periods (1-13 weeks)?**  
   *Your answer here.*

3. **How does the scalogram compare to simply looking at the raw return series?**  
   *Your answer here.*

### Solution 4 (reference)

> Attempt all tasks before reading the solution below.

In [ ]:
# ── SOLUTION 4 ────────────────────────────────────────────────

# 4a: Plot returns
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(dates, returns, color=COLORS['blue'], lw=0.8, alpha=0.8)
ax.axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
ax.set_xlabel('Date')
ax.set_ylabel('Log Return')
ax.set_title('Weekly Log-Returns')
plt.tight_layout()
plt.show()

# 4b: CWT scalogram
if HAS_PYWT:
    wavelet    = 'cmor1.5-1.0'

    # Frequency range: 4 to 104 weeks (in cycles/year)
    freqs_fin  = np.geomspace(52/104, 52/4, 60)
    scales_fin = pywt.frequency2scale(wavelet, freqs_fin)

    coeffs_fin, freqs_cwt_fin = pywt.cwt(returns, scales_fin, wavelet,
                                          sampling_period=1/52)
    power_fin   = np.abs(coeffs_fin) ** 2
    periods_fin = 1.0 / freqs_cwt_fin    # years

    fig, axes = plt.subplots(2, 1, figsize=(14, 9),
                              gridspec_kw={'height_ratios': [1, 2.5]})

    # Top: return series
    axes[0].plot(dates, returns, color=COLORS['blue'], lw=0.8, alpha=0.8)
    axes[0].axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
    axes[0].set_ylabel('Log Return')
    axes[0].set_title('Weekly Returns and Wavelet Power Spectrum')
    axes[0].set_xlim(dates[0], dates[-1])

    # Bottom: scalogram (periods in weeks)
    im = axes[1].contourf(dates,
                          np.log2(periods_fin * 52),   # convert years to weeks
                          np.log2(power_fin + 1e-10),
                          levels=20, cmap='YlOrRd')
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Period (weeks, log2 scale)')

    yticks_w = [4, 8, 13, 26, 52, 104]
    axes[1].set_yticks(np.log2(yticks_w))
    axes[1].set_yticklabels([str(w) + 'w' for w in yticks_w])

    plt.colorbar(im, ax=axes[1], label='log2(Power)')
    plt.tight_layout()
    plt.show()

    print('Key observations from the scalogram:')
    print('  1. High power at short periods (4-13 weeks) during crisis episodes')
    print('     => Elevated short-run volatility during stress periods')
    print('  2. Power is clustered in time, consistent with GARCH volatility clustering')
    print('  3. The scalogram pinpoints WHEN elevated volatility occurred at each frequency,')
    print('     which the raw return series shows less clearly')
    print('  4. Longer periods (26-104 weeks) show more diffuse but persistent power')
else:
    print('PyWavelets not installed. Install with: pip install PyWavelets')

---
## Seminar Summary

You have practiced the main tools of frequency-domain time series analysis:

| Exercise | Method | Key result |
|----------|--------|------------|
| 1 — Sunspots | Periodogram, Welch | ~11-year solar cycle at f ~0.091 cycles/year |
| 2 — GDP | HP, BK, CF filters | Business cycle: ~20-quarter dominant period |
| 3 — Coherence | Squared coherence, phase | Investment lags GDP by ~3 quarters |
| 4 — Finance | Wavelet CWT, scalogram | Volatility clustering visible in time-frequency plane |

### Common Pitfalls

- **Periodogram inconsistency**: Always smooth before inference (Welch, multitaper)
- **HP filter leakage**: HP passes low-frequency noise; BK/CF are more precise
- **BK edge effects**: K=12 removes 12 observations from each end — check your sample size
- **Wavelet cone of influence**: Edge effects grow larger at low frequencies (long periods)
- **Coherence interpretation**: High coherence does not imply causality

### Next Steps
- Long-memory estimation via spectral regression (GPH, Whittle)
- Wavelet coherence between two non-stationary series
- Spectral change-point detection for structural break analysis